# Lab 10 - Random Forest: Classification and Regression

This notebook applies **Lab 10 - Random Forest** to hazardous-event classification and PM2.5 regression.


## Lab 10 concepts used

- Train a random-forest classifier.
- Train a random-forest regressor.
- Compare classification and regression metrics.
- Inspect feature importances from ensemble trees.

The hazardous-event classifier excludes `European_AQI` from its inputs.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'AML Assignment' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT


In [ ]:
DATASET_FILENAME = 'global_urban_smog_pm25_hourly.csv'
matches = sorted((PROJECT_ROOT / 'Datasets').glob(f'*/{DATASET_FILENAME}'))
if not matches:
    raise FileNotFoundError(f'Could not find {DATASET_FILENAME} under {PROJECT_ROOT / "Datasets"}')
DATASET_PATH = matches[0]
data = pd.read_csv(DATASET_PATH)
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values(['Timestamp', 'City']).reset_index(drop=True)
print(DATASET_PATH)
data.head()


In [ ]:
def add_time_features(df):
    out = df.copy()
    out['Timestamp'] = pd.to_datetime(out['Timestamp'])
    out = out.sort_values(['Timestamp', 'City']).reset_index(drop=True)
    out['hour'] = out['Timestamp'].dt.hour
    out['dayofweek'] = out['Timestamp'].dt.dayofweek
    out['month'] = out['Timestamp'].dt.month
    out['dayofyear'] = out['Timestamp'].dt.dayofyear
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    return out

def chronological_split(df, train_size=0.8):
    split_idx = int(len(df) * train_size)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def latest_rows(df, max_rows):
    if len(df) <= max_rows:
        return df.copy()
    return df.tail(max_rows).copy()

NUMERIC_NO_AQI = [
    'Latitude', 'Longitude', 'PM10_ug_m3', 'PM2_5_ug_m3',
    'Carbon_Monoxide_ug_m3', 'Nitrogen_Dioxide_ug_m3',
    'Ozone_ug_m3', 'Dust_ug_m3', 'UV_Index',
    'hour', 'dayofweek', 'month', 'is_weekend'
]
CATEGORICAL_FEATURES = ['City']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocess, NUMERIC_NO_AQI),
        ('cat', categorical_preprocess, CATEGORICAL_FEATURES),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)


In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, precision_recall_curve, f1_score
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

classification_df = latest_rows(add_time_features(data), 80000)
train_df, test_df = chronological_split(classification_df, train_size=0.8)
X_train = train_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_train = train_df['Hazardous_Event']
X_test = test_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_test = test_df['Hazardous_Event']

rf_classifier = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(n_estimators=120, max_depth=14,
                                     min_samples_leaf=20,
                                     class_weight='balanced_subsample',
                                     n_jobs=-1, random_state=42))
])
rf_classifier.fit(X_train, y_train)
rf_pred = rf_classifier.predict(X_test)
print(classification_report(y_test, rf_pred, digits=3))


In [ ]:
feature_names = rf_classifier.named_steps['preprocess'].get_feature_names_out()
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_classifier.named_steps['model'].feature_importances_,
}).sort_values('importance', ascending=False)
importance_df.head(15)


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df.head(12), x='importance', y='feature')
plt.title('Random forest classifier feature importance')
plt.show()


In [ ]:
def add_time_features(df):
    out = df.copy()
    out['Timestamp'] = pd.to_datetime(out['Timestamp'])
    out = out.sort_values(['Timestamp', 'City']).reset_index(drop=True)
    out['hour'] = out['Timestamp'].dt.hour
    out['dayofweek'] = out['Timestamp'].dt.dayofweek
    out['month'] = out['Timestamp'].dt.month
    out['dayofyear'] = out['Timestamp'].dt.dayofyear
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    return out

def chronological_split(df, train_size=0.8):
    split_idx = int(len(df) * train_size)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def latest_rows(df, max_rows):
    if len(df) <= max_rows:
        return df.copy()
    return df.tail(max_rows).copy()

REGRESSION_NUMERIC_NO_AQI = [
    'Latitude', 'Longitude', 'PM10_ug_m3', 'Carbon_Monoxide_ug_m3',
    'Nitrogen_Dioxide_ug_m3', 'Ozone_ug_m3', 'Dust_ug_m3',
    'UV_Index', 'hour', 'dayofweek', 'month', 'is_weekend'
]
CATEGORICAL_FEATURES = ['City']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocess, REGRESSION_NUMERIC_NO_AQI),
        ('cat', categorical_preprocess, CATEGORICAL_FEATURES),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error


In [ ]:
regression_df = latest_rows(add_time_features(data), 80000)
train_df, test_df = chronological_split(regression_df, train_size=0.8)
X_train = train_df[REGRESSION_NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_train = train_df['PM2_5_ug_m3']
X_test = test_df[REGRESSION_NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_test = test_df['PM2_5_ug_m3']

rf_regressor = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(n_estimators=120, max_depth=16,
                                    min_samples_leaf=20, n_jobs=-1,
                                    random_state=42))
])
rf_regressor.fit(X_train, y_train)
reg_pred = rf_regressor.predict(X_test)
print('MAE:', mean_absolute_error(y_test, reg_pred))
print('RMSE:', root_mean_squared_error(y_test, reg_pred))
print('R2:', r2_score(y_test, reg_pred))


## What was learned from Lab 10

Random forests are strong non-linear baselines for both assignment tasks and provide useful feature-importance evidence for the report. They are less transparent than a single tree but usually more stable.
